In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

import torchvision
from torchvision.datasets import CIFAR10

# Datasets and DataLoaders

In [ ]:
from torch.utils.data import DataLoader
import torchvision.transforms as transforms


#image ==> scale(0,1) ==> normalize (-1,+1)

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))
])

trainset = CIFAR10(root="./data",train=True,download=True,transform=transform)
testset = CIFAR10(root="./data",train=False,download=True,transform=transform)

In [3]:
train_loader = DataLoader(trainset,batch_size = 64 , shuffle = True)
test_loader = DataLoader(testset,batch_size = 64)

# Build the CNN

In [4]:
class CNN(nn.Module):
  def __init__(self):
    super(CNN,self).__init__()

    self.conv_layers  = nn.Sequential(
        nn.Conv2d(3,32,kernel_size=3,padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2,2),  #kernel_size = 2 ,stride = 2

        nn.Conv2d(32,64,kernel_size=3,padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2,2),

        nn.Conv2d(64,128,kernel_size=3,padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2,2),
    )

    self.fc_layers = nn.Sequential(
        nn.Linear(4*4*128,256),
        nn.ReLU(),
        nn.Linear(256,10)
    )

  def forward(self,x):
    x = self.conv_layers(x)
    x = x.view(x.size(0),-1) # flattening
    x = self.fc_layers(x)
    return x

In [5]:
model = CNN()

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(model.parameters())


# Train the model

In [6]:
model.train()
epochs = 10
for epoch in range(epochs):
  epoch_training_loss = 0.0

  for images,labels in train_loader:
    optimizer.zero_grad()
    output = model.forward(images)
    loss = criterion(output,labels)
    loss.backward()
    optimizer.step()
    epoch_training_loss += loss.item()


  print(f"epoch : {epoch+1} & loss : {epoch_training_loss/len(train_loader)}")


epoch : 1 & loss : 1.3907029083012925
epoch : 2 & loss : 0.9546042407488884
epoch : 3 & loss : 0.7689019388631176
epoch : 4 & loss : 0.6466944742080806
epoch : 5 & loss : 0.5451153858619577
epoch : 6 & loss : 0.4528116788476934
epoch : 7 & loss : 0.36356732541757164
epoch : 8 & loss : 0.2861830704581097
epoch : 9 & loss : 0.22569355892632015
epoch : 10 & loss : 0.17332751267706342


# Evaluate the model

In [7]:
model.eval()
correct_labels = 0
total_labels = 0

with torch.no_grad():
  for images,labels in test_loader:
    outputs = model.forward(images)
    _,predicted = torch.max(outputs,1)
    correct_labels += (predicted == labels).sum().item()
    total_labels += labels.size(0)

print(f"Accuracy Score : {correct_labels/total_labels}")


Accuracy Score : 0.738
